# DS2002 · Capstone Analysis Clinic

**Lecture — 2026-11-30 · Fall 2026**  
**Class time:** 45 minutes

---

## Capstone analysis clinic

One week to the presentation. Analysis should be underway on all five questions; today is for the two that teams reliably get wrong, plus whatever is blocking you.

Bring your notebook. If your cleaning is not finished, say so early in the hour — that is a different conversation and it needs to happen now, not Friday.

### Question 5 — the one with a real trap in it

Start from the inventory table. A sellout is not a demand measurement, and reporting it as one is the mistake that costs the most marks on this project.

In [ ]:
import pandas as pd, sqlite3, os

def find_data(filename, folders=('data', '../data', '/kaggle/input', '/content')):
    for folder in folders:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    return None

db = find_data('inventory_and_sales.db')
if db:
    conn = sqlite3.connect(db)
    inv = pd.read_sql_query('SELECT * FROM inventory_levels', conn)
    print(inv.head())
    print()
    print('columns:', inv.columns.tolist())
else:
    print('database not found -- upload the data/ folder')
    inv = None

The test for a sellout, written so it works on your own frame. The `sell_through` column is what you actually report — anything at or above 100% is censored data.

In [ ]:
# Adapt the column names to whatever your database uses
example = pd.DataFrame({
    'vendor_id': ['V-18', 'V-18', 'V-01', 'V-05'],
    'item': ['Rain Poncho'] * 2 + ['Cheeseburger', 'Chicken Tacos'],
    'units_sold': [95, 40, 180, 210],
    'on_hand_start': [95, 120, 400, 215],
})
example['sell_through'] = (100 * example['units_sold']
                          / example['on_hand_start']).round(1)
example['censored'] = example['sell_through'] >= 99
print(example)
print()
print('rows where demand is a floor, not a measurement:',
      int(example['censored'].sum()))

Two of those four sold essentially their whole stock. For those rows the honest statement is "demand was at least this high." That is not a weaker finding — it is a stronger argument for increasing stock, and it is the difference between a recommendation somebody trusts and one they poke a hole in during questions.

### Question 4 — distance and zone, without fooling yourself

Two things go wrong here. Teams compare raw revenue across zones of wildly different size, and teams draw a trend through four points.

Normalize by whatever makes the zones different, and report the group sizes next to the numbers.

In [ ]:
zones = pd.DataFrame({
    'zone': ['A', 'B', 'C', 'D'],
    'revenue': [4800, 1900, 1200, 260],
    'capacity': [12000, 4000, 1500, 300],
    'orders': [1420, 610, 380, 41],
})
zones['revenue_per_capacity'] = (zones['revenue'] / zones['capacity']).round(3)
zones['avg_ticket'] = (zones['revenue'] / zones['orders']).round(2)
print(zones.sort_values('revenue_per_capacity', ascending=False))
print()
print('Zone D has 41 orders. Any per-order statistic there is noise, and it should')
print('be reported with its n or excluded with a stated reason.')

### The pre-presentation audit

Run this against your own notebook this week. Every one of these has cost a team marks before.

In [ ]:
audit = {
    'every merge has validate= or a row-count check': False,
    'no fillna(0) on a value that was actually missing': False,
    'every reported multiple states its baseline window': False,
    'every correlation states its n': False,
    'weather rows verified for all 6 game dates': False,
    'sellouts identified and reported as floors': False,
    'decision log present and current': False,
    'notebook runs clean from a fresh kernel': False,
}

for item, done in audit.items():
    print('OK ' if done else 'NO ', '-', item)
print()
todo = [k for k, v in audit.items() if not v]
print(f'{len(todo)} item(s) to fix before Monday')
if todo:
    print('start with:', todo[0])

Wednesday is presentation prep, Friday is the last graded checkpoint, and the presentation is the following Monday. Anything still broken on Friday will be broken in the presentation.